# 연구 A: 결과 확인과 후속 실험 설계

학습을 실행하지 않는 분석 전용 노트북입니다. CSV 읽기·설계 확인만 합니다. 자세한 방법은 `EPOCH_AND_OPTIMIZER_PLAN_KO.md`에 있습니다. GitHub 저장소를 내려받아 repo 루트 또는 이 노트북 폴더에서 실행하세요. Colab은 함께 제공한 문서/CSV 폴더를 먼저 준비해야 합니다.


In [ ]:
from pathlib import Path
import csv, json
candidates = [Path.cwd(), Path.cwd() / 'docs/research_a_20261016']
ROOT = next((p for p in candidates if (p / 'TIME_ESTIMATES.json').exists()), None)
assert ROOT is not None, '문서 폴더 또는 저장소 루트에서 실행하세요.'
with (ROOT / 'results/MODEL_COMPARISON.csv').open(encoding='utf-8-sig') as f:
    results = list(csv.DictReader(f))
assert len(results) == 8 and all(r['status'] == 'PASS' for r in results)
[(r['dataset'], r['model'], float(r['mAP50_95_percent']), float(r['delta_vs_same_environment_v8_mAP50_95_pp'])) for r in results]


## E1: 첫 100 epoch를 보존하는 추가 100 epoch

SGD·lr0/lrf·warmup을 유지합니다. LR horizon=100, mosaic off=91번째 epoch, YOLO26 loss horizon=100, 완료 예산=200, patience=0을 제안합니다. 100 epoch 수치 상태 불일치 시 101 진입 전에 중단합니다. 이 노트북은 제안 설정을 읽기만 합니다.


In [ ]:
proposal = json.loads((ROOT / 'configs/epoch_extension_proposal.json').read_text(encoding='utf-8'))
assert proposal['status'].startswith('DESIGN_ONLY')
proposal


In [ ]:
estimates = json.loads((ROOT / 'TIME_ESTIMATES.json').read_text(encoding='utf-8'))
[(e['dataset'], round(e['estimated_200epoch_total_seconds']/60, 2), e['planning_range_minutes']) for e in estimates]


## 모든 클래스 확인

취약 클래스만 보고 결론을 내리지 않습니다. 아래는 기존 8개 실행의 전체 클래스 AP와 동일 환경 v8 대비 변화입니다.


In [ ]:
with (ROOT / 'results/CLASS_AP_COMPARISON.csv').open(encoding='utf-8-sig') as f:
    class_results = list(csv.DictReader(f))
class_results


## E2: AdamW는 별도 100 epoch 단계

SGD 대조군은 기존 100 epoch입니다. 초기 제안은 동일한 수치 LR이며, AdamW 최적값을 뜻하지 않습니다. LR을 함께 바꾸면 별도 비교로 분리합니다. E1 결과를 보고 epoch를 임의 선택하지 않습니다. 새로운 실행 코드·학습 명령은 이 노트북에 없습니다.

## 연구 메모

- E1 승인 여부: 미정
- 구현/검증 요구사항: 계획서 참조
- AdamW 수치 사전 확정: 미정
- 추가 seed 반복: 미정
- K8/test: 제외
